In [1]:
import os
import json
import re
import pandas as pd

pd.options.display.max_columns = 10

In [2]:
def extract_json(response: str):
    """Extract JSON content from a formatted string."""
    match = re.search(r"```json\s*(.*?)\s*```", response, re.DOTALL)
    if match:
        json_str = match.group(1)
    else:
        json_str = response.strip('```json').strip('```')

    try:
        return json.loads(json_str)
    except json.JSONDecodeError as e:
        print(f"Chyba při dekódování JSON: {e}")
        return None

In [3]:
def process_txt_files(folder_path, prefix):
    """Zpracuje všechny txt soubory začínající prefixem (např. 'bank_part') ve složce a vrátí Pandas DataFrame."""
    all_data = []

    # Get files with prefix and end with .txt
    files = [f for f in os.listdir(folder_path) if f.startswith(prefix) and f.endswith(".txt")]

    # Order by number
    files.sort(key=lambda x: int(re.search(r'chunk(\d+)', x).group(1)))

    for filename in files:
        file_path = os.path.join(folder_path, filename)
        with open(file_path, "r", encoding="utf-8") as file:
            for line in file:
                try:
                    json_obj = json.loads(line.strip())
                    content_str = json_obj.get("response", {}).get("body", {}).get("choices", [{}])[0].get("message", {}).get("content", "")
                    extracted_json = extract_json(content_str)

                    if extracted_json:
                        row = {"id": json_obj["id"], "custom_id": json_obj["custom_id"]}
                        for feature in extracted_json.get("features", []):
                            row[feature["feature_name"]] = feature["answer"]

                        all_data.append(row)
                    else:
                        print(filename)
                except json.JSONDecodeError:
                    print(f"Chyba dekódování JSON v souboru {filename}")

    df = pd.DataFrame(all_data)
    return df

In [55]:
# Použití skriptu
#
folder_path = "../../data/outputs/hazard"
df = process_txt_files(folder_path, "hazard")

# Zobrazení výsledného dataframe
df.shape

(6212, 12)

# Merge with target

In [56]:
import requests
import pandas as pd
from io import StringIO

# URLs for the files
urls = [
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_train.csv",
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_valid.csv",
    "https://raw.githubusercontent.com/food-hazard-detection-semeval-2025/food-hazard-detection-semeval-2025.github.io/refs/heads/main/data/incidents_test.csv"
]

# Load each file into a DataFrame
dataframes = []
for url in urls:
    response = requests.get(url)
    response.raise_for_status()  # Raise an error for bad status codes
    csv_data = StringIO(response.text)  # Convert text to a file-like object
    df = pd.read_csv(csv_data)
    dataframes.append(df)

# Access the DataFrames
train_df, valid_df, test_df = dataframes

# Example: Display the first few rows of the training DataFrame
print(train_df.head())

   Unnamed: 0  year  month  day country  ...  \
0           0  1994      1    7      us  ...   
1           1  1994      3   10      us  ...   
2           2  1994      3   28      us  ...   
3           3  1994      4    3      us  ...   
4           4  1994      7    1      us  ...   

                                                text hazard-category  \
0  Case Number: 024-94   \n            Date Opene...      biological   
1  Case Number: 033-94   \n            Date Opene...      biological   
2  Case Number: 014-94   \n            Date Opene...      biological   
3  Case Number: 009-94   \n            Date Opene...  foreign bodies   
4  Case Number: 001-94   \n            Date Opene...  foreign bodies   

               product-category                  hazard  \
0  meat, egg and dairy products  listeria monocytogenes   
1  meat, egg and dairy products            listeria spp   
2  meat, egg and dairy products  listeria monocytogenes   
3  meat, egg and dairy products        pla

In [57]:
valid_df.shape[0] + train_df.shape[0] + test_df.shape[0]

6644

In [50]:
df = df.drop(columns=["id", "custom_id"], errors='ignore')
df = pd.get_dummies(
        df, sparse=False, prefix_sep='_'
    )

In [48]:
df

,hazard_type,product_type,recall_scope,contaminant_type,recall_initiator,...,regulatory_body,product_batch,recall_cost,consumer_impact,recall_resolution
0,other hazard,other,national,other,regulatory agency,...,USDA,unknown,0,other,other
1,other hazard,other,national,other,regulatory agency,...,USDA,Batch001,10000,inconvenience,refund
2,other hazard,other,national,other,regulatory agency,...,USDA,Batch-001,50000,inconvenience,refund
3,other hazard,other,national,other,regulatory agency,...,USDA,Batch-001,10000,inconvenience,refund
4,other hazard,other,national,other,regulatory agency,...,USDA,Batch12345,50000,inconvenience,refund
...,...,...,...,...,...,...,...,...,...,...,...
7025,other hazard,other,national,other,manufacturer,...,FDA,Batch12345,50000,inconvenience,refund
7026,other hazard,snacks,national,other,manufacturer,...,other,PB202310,50000,inconvenience,refund
7027,foreign bodies,snacks,national,foreign object,manufacturer,...,CFIA,Batch123,5000,inconvenience,refund
7028,allergens,snacks,national,allergen,manufacturer,...,FDA,BATCH12345,50000,health effects,refund


In [53]:
# 1) Libraries
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

X_train = df.iloc[:len(train_df)].drop(columns=["custom_id", "id"], errors='ignore')
# X_valid = df.iloc[-1130:-565].drop(columns=["custom_id", "id"], errors='ignore')
X_test = df.tail(565).drop(columns=["custom_id", "id"], errors='ignore')

y_train = train_df['hazard-category']
y_test = test_df['hazard-category'].tail(565)
# y_valid = valid_df['hazard-category']

In [54]:
# 1) Libraries
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score


# ----------------------------------------------------------------
# 4) Definice modelu RandomForestClassifier
model = GradientBoostingClassifier(random_state=42)

# 5) Nastavení rozsahu parametrů pro RandomizedSearchCV
param_dist = {
    "n_estimators": [50, 100, 200],       # Počet stromů v lese
    "max_depth": [3, 5, 10, None],        # Maximální hloubka stromu
    "min_samples_split": [2, 5, 10],      # Minimální počet vzorků pro split
    "min_samples_leaf": [1, 2, 5],        # Minimální počet vzorků v listu
}

# 6) Konfigurace RandomizedSearchCV (n_iter a cv lze upravit dle potřeby)
random_search = RandomizedSearchCV(
    model,
    param_distributions=param_dist,
    n_iter=10,             # kolik náhodných kombinací parametrů prozkoumat
    cv=5,                  # 5-fold cross-validace
    scoring="f1_macro",    # metrika, dle které se bude model porovnávat
    random_state=42,
    n_jobs=-1,             # využití všech CPU jader pro rychlejší výpočet
    verbose=1
)

# 7) Trénink modelu s vyhledáváním nejlepších hyperparametrů
random_search.fit(X_train, y_train)

# 8) Vypsání nejlepších parametrů a skóre
print("Nejlepší parametry:", random_search.best_params_)
print("Nejlepší skóre na trénovací cross-validaci:", random_search.best_score_)


Fitting 5 folds for each of 10 candidates, totalling 50 fits


C:\Users\vojta\miniconda3\envs\llm-features\lib\site-packages\sklearn\model_selection\_split.py:805: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(


KeyboardInterrupt: 

In [51]:

# 9) Ověření na testovací sadě
best_model = random_search.best_estimator_  # získáme nejlepší nalezený model
y_pred = best_model.predict(X_test)

# 10) Vyhodnocení
print("Přesnost na testu:", accuracy_score(y_test, y_pred))
print("Classification report na testu:")
print(classification_report(y_test, y_pred, zero_division=0))

ValueError: The feature names should match those that were passed during fit.
Feature names unseen at fit time:
- affected_units
- consumer_advisory
- consumer_impact
- contaminant_type
- distribution_channel
- ...
Feature names seen at fit time, yet now missing:
- affected_units_0
- affected_units_1
- affected_units_10
- affected_units_100
- affected_units_1000
- ...


In [52]:
clf = GradientBoostingClassifier(random_state=42)
clf.fit(X_train, y_train)

ValueError: could not convert string to float: 'other hazard'

In [39]:
y_test_pred = clf.predict(X_test)
from sklearn.metrics import classification_report
print(classification_report(y_test, y_test_pred))

                                precision    recall  f1-score   support

                     allergens       0.34      0.41      0.37       365
                    biological       0.34      0.37      0.36       343
                      chemical       0.00      0.00      0.00        52
food additives and flavourings       0.00      0.00      0.00         4
                foreign bodies       0.10      0.07      0.08       111
                         fraud       0.05      0.04      0.05        75
                     migration       0.00      0.00      0.00         1
          organoleptic aspects       0.00      0.00      0.00        10
                  other hazard       0.00      0.00      0.00        26
              packaging defect       0.00      0.00      0.00        10

                      accuracy                           0.29       997
                     macro avg       0.08      0.09      0.09       997
                  weighted avg       0.26      0.29      0.27 

C:\Users\vojta\miniconda3\envs\llm-features\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vojta\miniconda3\envs\llm-features\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\vojta\miniconda3\envs\llm-features\lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} i

In [11]:
grid_rfc = {
 'max_depth': [10, 20, 30, 40, 45, 50],
 'max_features': ['log2', 'sqrt', 30],
 'min_samples_leaf': [30, 40, 50, 60, 70, 90],
 'min_samples_split': [20, 30, 50, 60, 90],
 'n_estimators': [ 100, 200, 400, 1000]}

In [12]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import recall_score, accuracy_score, f1_score, precision_score, mean_absolute_error, root_mean_squared_error, mean_squared_error
import shap